# 4장 실습. 질문 설계

Jev 에게 무엇을 어떻게 묻느냐가 확률을 바꾼다. 질문을 쓰는 습관을 하나씩 바꿔 가며, 8장의 평가셋 위에서 정확도와 보정이 얼마나 움직이는지 잰다.

| 절 | 바꾸는 것 | 평가 과제 |
|---|---|---|
| 4.1 | 같은 요청을 여러 번 보낸다 | 취소 문의(영어·한국어) |
| 4.2 | 질문 문장 — 이중 부정 대 주장 하나 | 취소 문의 |
| 4.3 | `criteria` 의 설명을 지운다 | positive, topic |
| 4.4 | `state` 에 판단과 무관한 글을 섞는다 | entails |
| 4.5 | `state` 를 JSON 에서 문자열 하나로 바꾼다 | entails |
| 4.6 | 질문을 영어로 쓴다 | positive |
| 4.7 | 질문 셋을 한 요청에 묶는다 | 한국어 러닝 예제 |

비교는 같은 항목 200개를 두 방식으로 잰 뒤 짝지어 차이를 본다. 차이의 95% 구간이 0 을 걸치면 「차이가 없다」가 아니라 「200개로는 구별되지 않는다」로 읽는다.

8장(`nb08_evaluation.ipynb`)이 만든 `data/eval_v1.jsonl` 이 있어야 한다. Jev 에 약 1,650건(0.02달러 안팎)을 보낸다.

In [2]:
import os, json, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
import koreanize_matplotlib  # noqa: F401
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
DATA, OUT = Path("data"), Path("out")
OUT.mkdir(exist_ok=True)
TASKS = ["topic", "entails", "positive", "similarity"]
COLOR = {"topic": "#2a78d6", "entails": "#eb6834", "positive": "#1baf7a", "similarity": "#eda100"}
pd.set_option("display.precision", 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
read_jsonl = lambda p: [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

assert (DATA / "eval_v1.jsonl").exists(), "평가셋이 없다. nb00_setup → nb08_evaluation 을 먼저 실행한다"

JEV_URL = os.getenv("JEV_URL", "https://api.typesafe.ai")
http = requests.Session()
http.mount(JEV_URL, HTTPAdapter(max_retries=Retry(total=4, backoff_factor=0.5, status_forcelist=[408, 429, 500, 502, 503, 504])))

def ask(state, questions, model="jev-latest"):
    r = http.post(f"{JEV_URL}/v1/systemone", timeout=60, headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
                  json={"model": model, "state": state, "questions": questions})
    r.raise_for_status()
    return r.json() | {"_latency_s": r.elapsed.total_seconds()}

choice = lambda instructions, criteria: {"type": "choice", "instructions": instructions, "criteria": criteria}
score = lambda instructions, levels: {"type": "score", "instructions": instructions, "criteria": levels}
noul = lambda instructions, true="", false="": {"type": "noul", "instructions": instructions} | ({"criteria": {"true": true, "false": false}} if true else {})

채점과 비교는 8장의 함수를 옮겨 쓴다. `run()` 은 항목마다 Jev 에 묻고 채점해 기록을 만들고, `paired()` 는 같은 항목을 두 방식으로 잰 기록을 짝지어 비교한다.

In [3]:
def probs_of(answer):
    return answer.get("probabilities") or {"true": answer["noul"], "false": 1 - answer["noul"]}

def judge(item, resp):
    (task, q), = item["questions"].items()
    a = resp["answers"][task]
    pr = {str(k): float(v) for k, v in probs_of(a).items()}
    best = max(pr, key=pr.get)
    pred = {"noul": lambda: pr["true"] >= 0.5, "choice": lambda: best, "score": lambda: int(best)}[q["type"]]()
    return {"id": item["id"], "task": item["task"], "qtype": q["type"], "gold": item["gold"], "model": resp["model"],
            "pred": pred, "probs": pr, "conf": pr[best], "conf_field": a.get("confidence"), "correct": bool(pred == item["gold"]),
            "input_tokens": resp["usage"]["input_tokens"], "latency_s": resp["_latency_s"]}

def run(items, engine=lambda it: ask(it["state"], it["questions"]), workers=4):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda it: judge(it, engine(it)), items))

arrays = lambda recs: (np.array([r["conf"] for r in recs]), np.array([r["correct"] for r in recs], float))

def ece(conf, ok, bins=15):
    idx = np.clip(np.searchsorted(np.linspace(0, 1, bins + 1), conf, side="left") - 1, 0, bins - 1)
    return float(sum(abs(conf[idx == i].mean() - ok[idx == i].mean()) * (idx == i).sum() for i in np.unique(idx)) / len(conf))

def paired(a, b, n_boot=2000, seed=0):
    # 같은 항목을 두 방식으로 잰 기록 a, b. 항목을 짝지어 복원 추출해 차이(b − a)의 95% 구간을 낸다
    ib = {r["id"]: r for r in b}
    (fa, ca), (fb, cb) = arrays(a), arrays([ib[r["id"]] for r in a])
    idx = np.random.default_rng(seed).integers(0, len(ca), (n_boot, len(ca)))
    d_acc = cb[idx].mean(1) - ca[idx].mean(1)
    d_ece = [ece(fb[i], cb[i]) - ece(fa[i], ca[i]) for i in idx]
    ci = lambda v: f"({np.percentile(v, 2.5):+.3f}, {np.percentile(v, 97.5):+.3f})"
    return {"정확도": f"{ca.mean():.3f} → {cb.mean():.3f}", "Δ정확도 (95%)": f"{cb.mean() - ca.mean():+.3f} {ci(d_acc)}",
            "ECE": f"{ece(fa, ca):.3f} → {ece(fb, cb):.3f}", "ΔECE (95%)": f"{ece(fb, cb) - ece(fa, ca):+.3f} {ci(d_ece)}",
            "답이 바뀐 항목": int((ca != cb).sum())}

items = read_jsonl(DATA / "eval_v1.jsonl")
by = {t: [i for i in items if i["task"] == t] for t in TASKS}
base = {t: run(by[t]) for t in ["topic", "entails", "positive"]}
pd.DataFrame({t: {"정확도": arrays(r)[1].mean(), "ECE": ece(*arrays(r))} for t, r in base.items()}).T

,정확도,ECE
topic,0.80,0.097
entails,0.98,0.069
positive,0.85,0.048


## 4.1 실행 간 분산

2장의 취소 질문을 다시 본다. 튜토리얼 저자는 2026년 9월 20일 `jev-1.13.0` 에서 영어 이중 부정 질문에 0.54, 주장 하나 + 기준 질문에 0.41 을 받았다고 적었다. 같은 요청을 연달아 여덟 번씩 보내 값이 얼마나 흔들리는지 잰다.

In [4]:
N = 8
msg_en = {"message": "I would rather not cancel, but I can't afford this anymore."}
msg_ko = {"message": "해지하고 싶지는 않은데, 이제 요금을 감당할 수가 없어요."}
CASES = [
    ("영어", msg_en, "이중 부정", noul("Is it not the case that the customer does not want to keep the plan?")),
    ("영어", msg_en, "주장 하나 + 기준", noul("Is the customer asking to cancel?",
                                          true="The customer says they are cancelling or wants to cancel",
                                          false="The customer wants to keep the plan or is only asking a question")),
    ("한국어", msg_ko, "이중 부정", noul("고객이 요금제를 유지하고 싶어 하지 않는다는 것은 사실이 아니다")),
    ("한국어", msg_ko, "주장 하나 + 기준", noul("고객이 해지를 요청한다", true="고객이 해지하겠다고 말하거나 해지를 원한다고 밝힌다",
                                            false="고객이 요금제를 유지하고 싶어 하거나 질문만 한다")),
]
ps = {(lang, name): [ask(msg, {"cancel": q})["answers"]["cancel"]["noul"] for _ in range(N)] for lang, msg, name, q in CASES}
pd.DataFrame([{"언어": lang, "질문": name, "평균": np.mean(v), "표준편차": np.std(v, ddof=1), "최소": min(v), "최대": max(v),
               "8회 값": " ".join(f"{p:.2f}" for p in v)} for (lang, name), v in ps.items()])

,언어,질문,평균,표준편차,최소,최대,8회 값
0,영어,이중 부정,0.604,0.052,0.49,0.65,0.49 0.60 0.60 0.65 0.64 0.65 0.60 0.60
1,영어,주장 하나 + 기준,0.464,0.028,0.42,0.51,0.48 0.46 0.42 0.51 0.44 0.47 0.45 0.48
2,한국어,이중 부정,0.847,0.007,0.84,0.86,0.85 0.85 0.84 0.85 0.85 0.86 0.84 0.84
3,한국어,주장 하나 + 기준,0.085,0.005,0.08,0.09,0.09 0.09 0.09 0.08 0.08 0.09 0.08 0.08


## 4.2 이중 부정과 주장 극성

위 표의 「이중 부정」과 「주장 하나 + 기준」 행이 이 습관의 비교다. 메시지가 「해지하고 싶지 않지만 감당이 안 된다」로 엇갈려 있으므로 정답은 정하지 않는다. 질문 문장에 따라 평균이 얼마나 옮겨 가는지, 여덟 번 사이의 흔들림이 질문마다 다른지를 본다. 정답이 있는 비교는 다음 절부터 평가셋 200개로 한다.

## 4.3 선택지 기준 명세

`criteria` 에는 선택지 이름과 설명을 함께 적었다. 설명을 지우고 이름만 남기면 어떻게 되는지 잰다. choice 는 설명 자리에 이름을 그대로 넣고, noul 은 참·거짓 기준을 통째로 지운다.

In [5]:
def strip_criteria(it):
    (qn, q), = it["questions"].items()
    bare_q = {"choice": lambda: q | {"criteria": {k: k for k in q["criteria"]}},
              "noul": lambda: {k: v for k, v in q.items() if k != "criteria"}}[q["type"]]()
    return it | {"questions": {qn: bare_q}}

bare = {t: run([strip_criteria(i) for i in by[t]]) for t in ["positive", "topic"]}
pd.DataFrame({t: paired(base[t], bare[t]) for t in bare}).T.rename_axis("기준 있음 → 기준 없음")

,정확도,Δ정확도 (95%),ECE,ΔECE (95%),답이 바뀐 항목
기준 있음 → 기준 없음,,,,,
positive,0.850 → 0.845,"-0.005 (-0.025, +0.010)",0.048 → 0.074,"+0.026 (-0.024, +0.054)",3
topic,0.800 → 0.770,"-0.030 (-0.070, +0.010)",0.097 → 0.107,"+0.010 (-0.026, +0.051)",18


## 4.4 상태 최소화와 패딩 효과

Jev 문서는 판단과 무관한 내용이 state 에 늘어나면 정확도가 떨어진다고 적었다. entails 의 state 에 평가셋에 없는 뉴스 제목 다섯 개를 `related_news` 로 덧붙인다. 질문은 그대로 `premise` 와 `hypothesis` 만 가리킨다.

In [6]:
used = {i["state"] for i in by["topic"]}
pool = [r["title"] for r in json.loads((DATA / "raw/ynat-v1.1_dev.json").read_text(encoding="utf-8")) if r["title"] not in used]

pad = lambda it, k=5: it | {"state": it["state"] | {"related_news": random.Random(it["id"]).sample(pool, k)}}   # 항목마다 고정된 방해 글
padded = run([pad(i) for i in by["entails"]])
pd.DataFrame({"필요한 것만 → 방해 글 5개": paired(base["entails"], padded)}).T

,정확도,Δ정확도 (95%),ECE,ΔECE (95%),답이 바뀐 항목
필요한 것만 → 방해 글 5개,0.980 → 0.970,"-0.010 (-0.025, +0.000)",0.069 → 0.070,"+0.001 (-0.012, +0.014)",2


## 4.5 구조화 상태: JSON 대 문자열

같은 내용을 `{"premise": ..., "hypothesis": ...}` 로 보내는 것과 문자열 하나로 이어 붙여 보내는 것을 비교한다. 문자열 쪽은 질문에서 백틱으로 필드를 가리킬 수 없으므로 「전제」·「가설」이라는 말로 가리킨다.

In [7]:
flat_q = {"entails": noul("전제가 사실이라면 가설도 반드시 사실이다", true="전제만 읽고도 가설이 참이라고 확정할 수 있다",
                          false="가설이 전제와 모순되거나, 전제만으로는 참인지 알 수 없다")}
flatten = lambda it: it | {"state": f"전제: {it['state']['premise']}\n가설: {it['state']['hypothesis']}", "questions": flat_q}
flat = run([flatten(i) for i in by["entails"]])
pd.DataFrame({"JSON + 백틱 → 문자열": paired(base["entails"], flat)}).T

,정확도,Δ정확도 (95%),ECE,ΔECE (95%),답이 바뀐 항목
JSON + 백틱 → 문자열,0.980 → 0.980,"+0.000 (-0.015, +0.015)",0.069 → 0.064,"-0.005 (-0.018, +0.010)",2


## 4.6 교차 언어 질문

Jev 문서는 영어가 주력이라고 밝혔다. 한국어 리뷰를 판단할 때 질문만 영어로 쓰면 나아질까. positive 의 질문과 기준을 영어로 옮겨 잰다. state 는 한국어 그대로다.

In [8]:
en_q = {"positive": noul("The reviewer rates this movie positively", true="The review praises, recommends or expresses satisfaction",
                         false="The review criticizes, expresses disappointment or does not recommend")}
english = run([i | {"questions": en_q} for i in by["positive"]])
pd.DataFrame({"한국어 질문 → 영어 질문": paired(base["positive"], english)}).T

,정확도,Δ정확도 (95%),ECE,ΔECE (95%),답이 바뀐 항목
한국어 질문 → 영어 질문,0.850 → 0.845,"-0.005 (-0.015, +0.000)",0.048 → 0.057,"+0.009 (-0.031, +0.050)",1


## 4.7 질문 배칭

2장의 러닝 예제는 질문 셋을 한 요청에 묶었다. 같은 질문을 셋으로 나눠 보내면 비용·지연·답이 어떻게 달라지는지 본다.

In [9]:
TICKET_STATE = {"channel": "앱 문의",
                "message": "결제가 두 번 됐어요. 어제 저녁에 한 번만 결제했는데 카드 문자가 두 번 왔습니다. 오늘 안에 처리 안 되면 해지할게요."}
TICKET_QUESTIONS = {
    "department": choice("이 문의를 처리할 팀", {"billing": "결제, 환불, 청구 오류", "technical": "앱 오류, 로그인, 연동 문제",
                                             "account": "가입, 해지, 요금제 변경", "general": "그 밖의 문의"}),
    "urgency": score("얼마나 급하게 처리해야 하는가", ["일반 처리로 충분하다", "오늘 안에 처리해야 한다", "즉시 사람이 연락해야 한다"]),
    "churn_risk": noul("고객이 해지를 언급하거나 해지할 뜻을 보인다",
                       true="해지·탈퇴·이용 중단을 직접 말하거나 조건부로 예고한다", false="해지에 관한 언급이 없다"),
}
together = ask(TICKET_STATE, TICKET_QUESTIONS)
apart = {n: ask(TICKET_STATE, {n: q}) for n, q in TICKET_QUESTIONS.items()}
brief = lambda a: a.get("choice") or round(a.get("score", a.get("noul")), 2)
pd.DataFrame([{"방식": "한 요청에 셋", "입력 토큰": together["usage"]["input_tokens"], "지연(초)": together["_latency_s"],
               **{n: brief(together["answers"][n]) for n in TICKET_QUESTIONS}},
              {"방식": "셋으로 나눔", "입력 토큰": sum(r["usage"]["input_tokens"] for r in apart.values()),
               "지연(초)": sum(r["_latency_s"] for r in apart.values()), **{n: brief(apart[n]["answers"][n]) for n in TICKET_QUESTIONS}}])

,방식,입력 토큰,지연(초),department,urgency,churn_risk
0,한 요청에 셋,565,0.173,billing,1.06,0.98
1,셋으로 나눔,1211,0.583,billing,1.07,0.99


## 정리

이 노트북이 잰 것이다. 결과 해석은 4장 본문에서 한다.

- 4.1 같은 요청을 여덟 번 보냈을 때의 평균·표준편차·최소·최대
- 4.2 같은 메시지에 대한 이중 부정 질문과 주장 하나 + 기준 질문의 차이
- 4.3~4.6 질문·state 를 한 가지씩 바꿨을 때 평가셋 200개에서의 정확도·ECE 변화와 95% 구간
- 4.7 질문 셋을 묶었을 때와 나눴을 때의 입력 토큰·지연·답